# Photodiode latency: several recordings side by side

Multi-recording version of `photodiode_latency_test.ipynb`. Every recording goes through
the same analysis as in that notebook; plots are shown side by side (one column per
recording) and compared at the end. Recordings made with the timestamp model also get an
LSL-timestamp analysis (cells 7 and 8).

Recordings are read from `data/EEG`. The subfolder `unknown config slx` is skipped.

**Computers.** Computer B (PsychoPy, `run_photodiode_latency_v0.py`) sends the LSL markers
and shows the flashes. Computer A (Simulink + g.HIamp at 600 Hz) records EEG, the photodiode
(g.TRIGbox digital input) and the markers.

## Changes tested so far, and how they appear in the file names

Settings that differ between recordings are read from the file name (cell 1,
`CONFIG_PATTERNS` and `FLAG_PATTERNS`).

| # | Change | Where | File-name tag | Recordings |
|---|---|---|---|---|
| 1 | Stimulus display set to 60 Hz in Windows (PsychoPy could only flip at 60 Hz at the 120 Hz setting) | Windows display settings, computer B; `expected_refresh_hz` 59.94 in the run dialog | none (see `*_run_info.txt` in `data/`) | all |
| 2 | Simulink frame length 4, 6, 8 | g.HIamp block configuration, computer A | `slx_framelength_X` (or `slx_frame_length_X`) | 2026-10-07_3, 2026-10-08_1..3 |
| 3 | LSL keepalive counts samples against the clock, so it really sends 600 samples/s (before: about 491/s at the 600 setting) | `lsl_trigger.py`, commit "Fixed the LSL sync issue" | none | 2026-10-08 recordings |
| 4 | Timestamp model: second LSL inlet (`LSLMarkerTimestamps` MATLAB System block) saves marker value, rising-edge code, B timestamp, B→A clock offset, A clock per step and samples pulled per step | `matlab/on_eeg_computer/HIamp_record_LSL_DD_vMMR.slx` + `LSLMarkerTimestamps.m` (until 13 Oct in `timestamped_LSL_model/`); LSL Receive block at 600 Hz | `timestamped_LSL` + a companion `*_LSLtiming.mat` | 2026-10-08_4_* |
| 5 | Baseline of the timestamp model: PsychoPy started right after the Simulink model | — | `start_0` | 2026-10-08_4 |
| 6 | PsychoPy started 1 minute after the Simulink model | — | `start_1mindelay` | 2026-10-08_4 |
| 7 | Workspace logging off (Time, Output, Signal logging, Data stores unchecked; `To File` still records) | Model Settings → Data Import/Export | `workspacelogging_off` | 2026-10-08_4 |
| 8 | Multithreaded co-simulation off, model and g.HIamp block (`set_param(..., 'MultiThreadCoSim', 'off')`) | MATLAB Command Window | `multithread_off` | 2026-10-08_4 |
| 9 | PsychoPy sender v1: LSL outlet created only after the window is open and the refresh rate measured; "start Simulink" prompt on the stimulus screen (SPACE); `expected_refresh_hz` 59.94 | `run_photodiode_latency_v1_trig_after_framerate_measurement.py`, computer B | `trig_after_framerate` (also `v1_trig`, `v1trig`) | 2026-10-08_4 |
| 11 | Experiment-start screen after the sensor alignment: SPACE sends marker **1** on the next flip; calibration and flashes are counted from it (all three latency scripts, from 2026-10-13) | `run_photodiode_latency_v0/v1/v2...py`, computer B | `expstart` | new |
| 12 | Calibration database: `run_photodiode_latency_database.py` (final configuration, square and sensor always at the screen centre, short = 100 or long = 500 flashes with the width ramp repeating every 100) | computer B | `sqcenter`, `short` / `long` | from 2026-10 |
| 10 | PsychoPy sender v2: LSL outlet created only after SPACE on the sensor-alignment screen; START (9) sent after Simulink is started | `run_photodiode_latency_v2_trig_after_space_input.py`, computer B | `trig_after_allignment_screen` (also `trig_after_alignment`, `v2_trig`, `v2trig`) | 2026-10-08_4 |

Runs 1-8 used `run_photodiode_latency_v0.py`, which creates the LSL outlet *before* opening
the window. The timestamp runs showed that this causes a ~0.8 s stall at the start of the
refresh-rate measurement that the Simulink model never catches up (cells 7-8). Changes 9 and
10 move the outlet creation later to remove it; their runs should show no backlog and
`lsl_resyncs` 0 (read from the PsychoPy `*_run_info.txt`).

When a recording contains marker 1, the analysis ignores everything before it (sensor
alignment, start-up) and reports the time from marker 1 to the first flash.

Note on 7 and 8: the timestamp model saved after these runs has
multithreading **off** but workspace logging **on**, so the `multithread_off` run was
probably recorded with logging switched back on (changes tested one at a time, not stacked).

Not tested as a separate run: a different LSL Receive `sampleRate` (it was already 600 Hz in
the timestamp model; the 2026-10-07 model file had 200 Hz).

# Cell 1: find and load the recordings

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import h5py
import scipy.io
import matplotlib.pyplot as plt

# Recordings to compare. Leave MAT_NAMES empty to use every .mat file in the EEG data
# folder (and its subfolders, except EXCLUDE_DIRS) whose name matches MAT_GLOB.
MAT_NAMES    = []
MAT_GLOB     = "*slx_frame*length_*.mat"      # matches slx_framelength_8 and slx_frame_length_8
EXCLUDE_DIRS = ["unknown config slx"]       # subfolders that are never analysed here

TIMING_SUFFIX = "_LSLtiming"                # companion file written by the timestamp model

# Settings that differ between recordings, read from the file name (see the table above).
# Numbers: name -> regex with one number group.
CONFIG_PATTERNS = {"slx_frame_length": r"slx_frame_?length_(\d+)"}
# On/off changes: name -> text (or tuple of alternative texts) that marks the change in
# the file name. The order of this list is also the order in which the runs were recorded.
FLAG_PATTERNS = {"timestamped_lsl":       "timestamped_LSL",
                 "start_1min_delay":      "start_1mindelay",
                 "workspace_logging_off": "workspacelogging_off",
                 "multithread_off":       "multithread_off",
                 "sender_v1":             ("v1_trig", "v1trig", "trig_after_framerate"),   # outlet after refresh measurement
                 "sender_v2":             ("v2_trig", "v2trig", "trig_after_allignment",   # outlet after alignment SPACE
                                           "trig_after_alignment", "trig_after_space"),
                 "exp_start_trigger":     ("expstart", "exp_start"),   # marker 1 after alignment
                 "square_center":         ("sqcenter",),               # database: square at screen centre
                 "square_corner":         ("sqcorner",),               # database: square bottom-right
                 "long_run":              ("_long",)}                  # database: 500-flash run
FLAG_LABELS = {"timestamped_lsl": "LSL ts", "start_1min_delay": "1 min delay",
               "workspace_logging_off": "logging off", "multithread_off": "multithread off",
               "sender_v1": "v1 trig after refresh", "sender_v2": "v2 trig after SPACE",
               "exp_start_trigger": "exp start", "square_center": "centre",
               "square_corner": "corner", "long_run": "long"}
SENDER_FLAGS = ("sender_v1", "sender_v2")   # PsychoPy versions with the stall fix
COMPARE_BY = None    # x axis of the comparison plots in cell 6: None = one position per
                     # recording, or a CONFIG_PATTERNS name such as "slx_frame_length"

# PsychoPy output (computer B), used to check that the timestamps match the sent markers
PSYCHOPY_GLOB = "*_photodiode_latency_latency_flashes.csv"

MARKER_CH   = -1       # last column  = LSL marker
PHOTO_CH    = -2       # second-last  = photodiode / g.TRIGbox digital input
FLASH_CODE, CAL_DARK, CAL_LIGHT, BLOCK_CODE = 20, 5, 6, 7
EXP_START_CODE = 1     # experiment start (after the alignment screen; scripts from 2026-10-13)
RAMP_CYCLE  = 100      # the flash-duration ramp repeats every 100 flashes (database long runs)
ON_START_MS = 100.0    # duration of flash 1, as set in the run dialog
ON_STEP_MS  = 1000 / 60  # increase per flash after rounding to frames (16.67 ms at 60 and 120 Hz)
MAX_GAP     = 0.030    # s; gaps shorter than this inside a pulse are bridged
MIN_PULSE   = 0.030    # s; pulses shorter than this are ignored
OUTLIER_MS  = 150      # paired delays further than this from the trend are left out

data_dir = next((p for p in (Path("../data/EEG"), Path("data/EEG")) if p.is_dir()), None)
assert data_dir is not None, "EEG data folder not found in ../data/EEG or data/EEG"
if MAT_NAMES:
    paths = [data_dir / n for n in MAT_NAMES]
    missing = [p.name for p in paths if not p.exists()]
    assert not missing, f"not found in {data_dir}: {missing}"
else:
    paths = sorted(p for p in data_dir.rglob(MAT_GLOB)
                   if not set(p.relative_to(data_dir).parts[:-1]) & set(EXCLUDE_DIRS)
                   and not p.stem.endswith(TIMING_SUFFIX))
    assert paths, f"no file matching {MAT_GLOB} in {data_dir.resolve()} (excluding {EXCLUDE_DIRS})"


def parse_config(name):
    cfg = {}
    for key, pattern in CONFIG_PATTERNS.items():
        m = re.search(pattern, name)
        cfg[key] = int(m.group(1)) if m else None
    for key, text in FLAG_PATTERNS.items():
        texts = (text,) if isinstance(text, str) else text
        cfg[key] = any(t in name for t in texts)
    return cfg


def run_order(r):
    """Date and run number from the file name (..._20261008_4_...), then the flag order."""
    m = re.search(r"_(\d{8})_(\d+)", r["path"].name)
    date_run = (m.group(1), int(m.group(2))) if m else ("", 0)
    flags = [i for i, k in enumerate(FLAG_PATTERNS) if r["cfg"][k] and k != "timestamped_lsl"]
    return date_run + (max(flags) + 1 if flags else 0, r["path"].name)


def load_eeg(path):
    if h5py.is_hdf5(path):                        # MATLAB v7.3 = HDF5
        with h5py.File(path, "r") as f:
            eeg = f["EEG_data"][()]
    else:
        eeg = scipy.io.loadmat(path)["EEG_data"]
    eeg = np.asarray(eeg, dtype=float)
    return eeg.T if eeg.shape[0] < eeg.shape[1] else eeg   # samples x channels


psychopy_dir = data_dir.parent                    # data/ holds the PsychoPy output
psychopy_csvs = {p: pd.read_csv(p) for p in sorted(psychopy_dir.glob(PSYCHOPY_GLOB))}

recs = []
for p in paths:
    eeg = load_eeg(p)
    t = eeg[:, 0]
    cols = []
    for c in range(eeg.shape[1] - 3, eeg.shape[1]):
        u = np.unique(eeg[:, c])
        desc = f"values {u.astype(int).tolist()}" if u.size <= 12 else f"{u.size} distinct values"
        cols.append(f"  col {c}: min {eeg[:, c].min():.4g}, max {eeg[:, c].max():.4g}, {desc}")
    timing = p.with_name(p.stem + TIMING_SUFFIX + p.suffix)
    recs.append({"path": p, "cfg": parse_config(p.name), "t": t,
                 "fs": 1.0 / np.median(np.diff(t)),
                 "marker": np.rint(eeg[:, MARKER_CH]).astype(int),
                 "photo_raw": eeg[:, PHOTO_CH].copy(),
                 "shape": eeg.shape, "col_desc": cols,
                 "timing_path": timing if timing.exists() else None})
    del eeg                                       # keep only the columns we need

recs.sort(key=run_order)                          # the order in which the runs were recorded

# short label per recording, e.g. "frame 8 | LSL ts | 1 min delay"; repeats get #1, #2, ...
for r in recs:
    parts = [f"frame {v}" for k, v in r["cfg"].items() if k in CONFIG_PATTERNS and v is not None]
    parts += [FLAG_LABELS.get(k, k) for k in FLAG_PATTERNS if r["cfg"][k]]
    if r["cfg"].get("timestamped_lsl") and not any(r["cfg"][k] for k in FLAG_PATTERNS if k != "timestamped_lsl"):
        parts.append("baseline")
    r["label"] = " | ".join(parts) if parts else r["path"].stem
counts = pd.Series([r["label"] for r in recs]).value_counts()
seen = {}
for r in recs:
    base = r["label"]
    if counts[base] > 1:
        seen[base] = seen.get(base, 0) + 1
        r["label"] = f"{base} #{seen[base]}"

overview = pd.DataFrame([{"file": r["path"].name, **r["cfg"],
                          "timing_file": r["timing_path"] is not None,
                          "samples": r["shape"][0], "columns": r["shape"][1],
                          "duration_s": r["t"][-1] - r["t"][0], "fs_hz": r["fs"],
                          "sample_ms": 1000 / r["fs"]} for r in recs],
                        index=[r["label"] for r in recs])
print(f"{len(recs)} recordings in {data_dir.resolve()} "
      f"({sum(r['timing_path'] is not None for r in recs)} with an LSL timing file); "
      f"{len(psychopy_csvs)} PsychoPy flash CSVs in {psychopy_dir.resolve()}\n")
print(overview.round(2).to_string())
for r in recs:
    print(f"\n{r['label']}  (last 3 columns)")
    print("\n".join(r["col_desc"]))

# Cell 2: analysis functions

Same steps as cells 2 and 3 of `photodiode_latency_test.ipynb`, written as functions so
they can run once per recording.

One addition: if a recording has exactly one photodiode pulse per flash marker and the
pulse widths follow the marker widths (r > 0.99), the n-th pulse is paired with flash n.
The width numbering of the original notebook is only used when that check fails; on
recordings with about ±6 ms of photodiode width noise it can mis-number the pulses.

`load_timing` and `analyze_timing` handle the `*_LSLtiming.mat` files of the timestamp
model; the docstring of `analyze_timing` explains each quantity.

In [ ]:
def pulses(mask, t, fs):
    """Runs of True in `mask` -> DataFrame(onset_s, width_ms). Short gaps are bridged, short runs dropped."""
    d = np.diff(np.r_[0, mask.astype(int), 0])
    on, off = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    if on.size == 0:
        return pd.DataFrame({"onset_s": [], "width_ms": []})
    keep = np.r_[True, (on[1:] - off[:-1]) / fs > MAX_GAP]        # start of a new pulse
    on, off = on[keep], off[np.r_[keep[1:], True]]
    ok = (off - on) / fs >= MIN_PULSE
    return pd.DataFrame({"onset_s": t[on[ok]], "width_ms": (off[ok] - on[ok]) / fs * 1000})


def ramp_score(p):                                # share of neighbours that differ by about one step
    dw = np.diff(p.width_ms)
    return np.mean((dw > 0.3 * ON_STEP_MS) & (dw < 1.8 * ON_STEP_MS)) if len(dw) else 0.0


def width_fit(idx, width):
    b, a = np.polyfit(idx, width, 1)
    return a, b


def number_in_order(x):
    """Whole flash numbers closest to x that also increase from one pulse to the next."""
    offs = np.arange(-2, 3)
    g = np.rint(x)[:, None] + offs[None, :]                     # candidate numbers per pulse
    cost = (g - x[:, None]) ** 2
    best, back = cost[0].copy(), np.zeros(g.shape, int)
    for j in range(1, len(x)):
        ok = g[j - 1][:, None] < g[j][None, :]                  # previous number must be smaller
        tot = best[:, None] + np.where(ok, 0.0, 25.0)           # out-of-order costs a penalty
        back[j] = tot.argmin(axis=0)
        best = tot.min(axis=0) + cost[j]
    path = [best.argmin()]
    for j in range(len(x) - 1, 0, -1):
        path.append(back[j][path[-1]])
    return g[np.arange(len(x)), path[::-1]].astype(int)


def trend(m):                                     # robust straight line through the delays
    keep = np.ones(len(m), bool)
    for _ in range(3):
        c = np.polyfit(m.onset_s_mk[keep], m.delay_ms[keep], 1)
        r = m.delay_ms - np.polyval(c, m.onset_s_mk)
        mad = 1.4826 * np.median(np.abs(r - np.median(r))) + 1e-9
        keep = (np.abs(r - np.median(r)) < 4 * mad).to_numpy()
    return c, r, mad


def detect(r):
    """Marker events and photodiode pulses (cell 2 of the single-recording notebook)."""
    t, fs, marker, photo_raw = r["t"], r["fs"], r["marker"], r["photo_raw"]
    codes = [c for c in np.unique(marker) if c != 0]
    assert codes, "the marker column has no nonzero values"
    ev = pd.concat([pulses(marker == c, t, fs).assign(code=c) for c in codes]) \
        .sort_values("onset_s").reset_index(drop=True)
    # experiment start (marker 1): everything before it (alignment, start-up) is ignored
    t_exp = ev.loc[ev.code == EXP_START_CODE, "onset_s"].min()
    if np.isfinite(t_exp):
        n_before = int((ev.onset_s < t_exp).sum())
        ev = ev[ev.onset_s >= t_exp].reset_index(drop=True)
        print(f"Experiment start (marker {EXP_START_CODE}) at {t_exp:.3f} s; "
              f"{n_before} earlier marker events ignored.")
    print(ev.groupby("code").agg(n=("code", "size"), first_s=("onset_s", "min"),
                                 width_min=("width_ms", "min"), width_max=("width_ms", "max")).round(1).to_string())
    mk = ev[ev.code == FLASH_CODE].reset_index(drop=True)
    assert len(mk), f"no flash markers (code {FLASH_CODE})"
    mk["flash"] = np.arange(1, len(mk) + 1)       # markers are complete, so order = flash number
    mk["ramp"] = (mk.flash - 1) % RAMP_CYCLE + 1      # position in the repeating width ramp
    mk["intended_ms"] = ON_START_MS + (mk.ramp - 1) * ON_STEP_MS
    t_block = ev.loc[ev.code == BLOCK_CODE, "onset_s"].min()
    if np.isnan(t_block):
        t_block = t[0]
        print(f"No flash-block marker {BLOCK_CODE}; using all photodiode pulses.")

    # photodiode: try both polarities and keep the one whose pulse widths form the rising ramp
    thr = (photo_raw.min() + photo_raw.max()) / 2
    photo_hi = photo_raw > thr
    max_w = mk.intended_ms.max() * 1.6

    def flash_pulses(mask):
        p = pulses(mask, t, fs)
        return p[(p.onset_s > t_block) & (p.width_ms <= max_w)].reset_index(drop=True)

    cands = {"HIGH": flash_pulses(photo_hi), "LOW": flash_pulses(~photo_hi)}
    scores = {k: ramp_score(v) for k, v in cands.items()}
    pol = max(scores, key=scores.get)
    pdp = cands[pol]
    print(f"\nRamp score (share of neighbouring pulses one step apart): "
          f"HIGH {scores['HIGH']:.0%} | LOW {scores['LOW']:.0%}")
    print(f"Polarity: square ON = photodiode {pol}")
    print(f"\nFlash markers: {len(mk)}   photodiode pulses in the flash block: {len(pdp)}")
    print(f"Marker widths:     {mk.width_ms.min():.0f} to {mk.width_ms.max():.0f} ms "
          f"(intended {mk.intended_ms.min():.0f} to {mk.intended_ms.max():.0f})")
    if len(pdp):
        print(f"Photodiode widths: {pdp.width_ms.min():.0f} to {pdp.width_ms.max():.0f} ms")
    real_s = (mk.intended_ms.sum() / 1000) + 0.5 * (len(mk) - 1)
    print(f"Flash block length: recorded {mk.onset_s.iloc[-1] - mk.onset_s.iloc[0]:.1f} s between first and last marker; "
          f"expected about {real_s - mk.intended_ms.iloc[-1] / 1000:.0f} s if the model keeps real time")
    if np.isfinite(t_exp):
        print(f"Experiment start -> first flash marker: {mk.onset_s.iloc[0] - t_exp:.3f} s")
    r.update(ev=ev, mk=mk, pdp=pdp, pol=pol, t_exp=t_exp,
             square_on=photo_hi if pol == "HIGH" else ~photo_hi)


def pair_by_order(mk, pdp):
    """One photodiode pulse per marker and the widths rise together -> the n-th pulse is flash n."""
    pairs = pdp.assign(flash=mk.flash.to_numpy()).merge(
        mk[["flash", "onset_s", "width_ms", "intended_ms"]], on="flash", suffixes=("_pd", "_mk"))
    pairs["delay_ms"] = (pairs.onset_s_pd - pairs.onset_s_mk) * 1000
    c, res, mad = trend(pairs)
    outliers = pairs[np.abs(res) > OUTLIER_MS]
    pairs = pairs[np.abs(res) <= OUTLIER_MS].reset_index(drop=True)
    b_p = width_fit((pairs.flash - 1) % RAMP_CYCLE, pairs.width_ms_pd)[1]
    return pairs, outliers, 0, b_p, len(pdp)


def pair_by_width(mk, pdp):
    """Number the photodiode pulses by width (cell 3 of the single-recording notebook).
    Used when pulses are missing or extra, so the order alone cannot be trusted."""
    pdp = pdp.copy()
    # photodiode: chain neighbouring pulses (each should be one step longer than the previous one)
    dw = np.diff(pdp.width_ms)
    near = dw[(dw > 0.3 * ON_STEP_MS) & (dw < 1.8 * ON_STEP_MS)]
    b0 = np.median(near) if near.size else ON_STEP_MS
    rel = np.r_[0, np.cumsum(np.rint(dw / b0))].astype(int)    # index relative to the first pulse
    a_p, b_p = width_fit(rel, pdp.width_ms)
    pdp["rel"] = number_in_order(((pdp.width_ms - a_p) / b_p).to_numpy())
    good = pdp[~pdp.rel.duplicated(keep=False)]

    # the first pulse is flash 1 + k; choose k so that the delay is most consistent across the run
    def pair(k):
        m = good.assign(flash=good.rel - good.rel.min() + 1 + k).merge(
            mk[["flash", "onset_s", "width_ms", "intended_ms"]], on="flash", suffixes=("_pd", "_mk"))
        m["delay_ms"] = (m.onset_s_pd - m.onset_s_mk) * 1000
        return m

    scan = []
    for k in range(-(good.rel.max() - good.rel.min()), len(mk) + 1):
        m = pair(k)
        if len(m) >= max(8, 0.5 * len(good)):
            scan.append((trend(m)[2], k, len(m)))
    assert scan, "No alignment found: photodiode widths do not overlap the marker ramp. Check the detect() output."
    scan.sort()
    best_spread, k_best, _ = scan[0]
    pairs = pair(k_best)

    # safety net: a width misread by one step shows up as a delay far from the trend -> try the neighbours
    c, res, mad = trend(pairs)
    fixed = 0
    for i in pairs.index[np.abs(res) > OUTLIER_MS]:
        alts = []
        for shift in (-2, -1, 1, 2):
            f = int(pairs.flash[i]) + shift
            if 1 <= f <= len(mk) and f not in set(pairs.flash.astype(int)):
                mt = mk.onset_s[f - 1]
                d_alt = (pairs.onset_s_pd[i] - mt) * 1000
                alts.append((abs(d_alt - np.polyval(c, mt)), f, mt, d_alt))
        if alts and min(alts)[0] < OUTLIER_MS:
            _, f, mt, d_alt = min(alts)
            pairs.loc[i, ["flash", "onset_s_mk", "width_ms_mk", "intended_ms", "delay_ms"]] = \
                [f, mt, mk.width_ms[f - 1], mk.intended_ms[f - 1], d_alt]
            fixed += 1
    c, res, mad = trend(pairs)
    outliers = pairs[np.abs(res) > OUTLIER_MS]
    pairs = pairs[np.abs(res) <= OUTLIER_MS].sort_values("flash").reset_index(drop=True)

    print(f"Photodiode pulses numbered by width: {len(good)}/{len(pdp)}")
    print(f"First photodiode pulse = flash {int(pairs.flash.min())}   (delay spread {best_spread:.2f} ms"
          + (f"; next-best alignment: {scan[1][0]:.1f} ms)" if len(scan) > 1 else ")"))
    return pairs, outliers, fixed, b_p, len(good)


def pair_flashes(r):
    """Pair every flash marker with its photodiode pulse.
    If there is exactly one pulse per marker and the pulse widths follow the marker
    widths (r > 0.99), the n-th pulse is flash n. Otherwise number the pulses by width."""
    mk, pdp = r["mk"], r["pdp"]
    assert len(pdp) >= 2, "fewer than 2 photodiode pulses in the flash block"
    b_m = width_fit(mk.ramp, mk.width_ms)[1]
    r_w = np.corrcoef(pdp.width_ms, mk.width_ms)[0, 1] if len(pdp) == len(mk) else np.nan
    if r_w > 0.99:
        method = "order"
        print(f"Pairing by order: {len(pdp)} photodiode pulses for {len(mk)} markers, "
              f"width correlation r = {r_w:.4f}")
        pairs, outliers, fixed, b_p, n_numbered = pair_by_order(mk, pdp)
    else:
        method = "width"
        why = (f"width correlation in order r = {r_w:.3f}" if np.isfinite(r_w)
               else f"{len(pdp)} photodiode pulses for {len(mk)} markers")
        print(f"Pairing by width numbering ({why})")
        pairs, outliers, fixed, b_p, n_numbered = pair_by_width(mk, pdp)
    missing = sorted(set(mk.flash) - set(pairs.flash.astype(int)))

    print(f"Paired flashes: {len(pairs)}/{len(mk)}   (renumbered by one or two steps: {fixed}; "
          f"left out as inconsistent: {len(outliers)})")
    print(f"Flashes without a photodiode pulse: {missing if missing else 'none'}")
    print(f"\nWidth increase per flash:  intended {ON_STEP_MS:.2f} ms | marker {b_m:.2f} ms | photodiode {b_p:.2f} ms")
    print(f"Time-base ratio  marker / intended: {b_m / ON_STEP_MS:.3f}    photodiode / intended: {b_p / ON_STEP_MS:.3f}"
          "    (1.000 = keeps real time)")
    r.update(pairs=pairs, outliers=outliers, missing=missing, fixed=fixed,
             b_m=b_m, b_p=b_p, n_numbered=n_numbered, method=method)


def summarize(r):
    """Latency, drift and jitter of the paired flashes (cell 4). Returns one row of results."""
    pairs, fs = r["pairs"], r["fs"]
    d = pairs.delay_ms
    x = pairs.onset_s_mk - pairs.onset_s_mk.iloc[0]
    slope, icpt = np.polyfit(x, d, 1)
    r["detr"] = d - (icpt + slope * x)
    print("\n=== delay = photodiode onset - marker onset of the SAME flash (positive: light after marker) ===")
    print(f"  LATENCY  mean {d.mean():9.2f} ms   median {d.median():9.2f} ms")
    print(f"  first 5 flashes {d.iloc[:5].mean():9.2f} ms   last 5 flashes {d.iloc[-5:].mean():9.2f} ms")
    print(f"  DRIFT    {slope * 60:+.2f} ms per minute ({d.iloc[-1] - d.iloc[0]:+.1f} ms from first to last paired flash)")
    print(f"  JITTER   SD {d.std():.2f} ms raw   |   SD {r['detr'].std():.2f} ms after removing the drift")
    print(f"  resolution: 1 sample = {1000 / fs:.2f} ms")
    return {"fs_hz": fs, "sample_ms": 1000 / fs, "polarity_on": r["pol"], "pairing": r["method"],
            "exp_start_s": r["t_exp"],
            "exp_start_to_first_flash_s": r["mk"].onset_s.iloc[0] - r["t_exp"],
            "flashes": len(r["mk"]), "paired": len(pairs), "missing": len(r["missing"]),
            "left_out": len(r["outliers"]),
            "latency_mean_ms": d.mean(), "latency_median_ms": d.median(),
            "latency_first5_ms": d.iloc[:5].mean(), "latency_last5_ms": d.iloc[-5:].mean(),
            "drift_ms_per_min": slope * 60,
            "jitter_sd_ms": d.std(), "jitter_sd_detrended_ms": r["detr"].std(),
            "delay_min_ms": d.min(), "delay_max_ms": d.max(),
            "timebase_marker": r["b_m"] / ON_STEP_MS, "timebase_photodiode": r["b_p"] / ON_STEP_MS}

# --- LSL timestamp analysis (timestamp model only) ------------------------------------
TS_COLS  = ["marker", "edge_code", "edge_ts_B", "offset", "t_A", "n_pulled"]
STALL_MS = 100       # a model step longer than this counts as a stall (normal step: 1.67 ms)


def load_timing(path, n):
    """Output of the LSLMarkerTimestamps block, saved by a To File block in Timeseries
    format (MATLAB v7.3). h5py cannot open the timeseries object itself, but its time
    vector (n values) and data array (n x 6) are stored as plain arrays under #refs#."""
    with h5py.File(path, "r") as f:
        refs = f["#refs#"]
        arrays = [refs[k][()] for k in refs
                  if isinstance(refs[k], h5py.Dataset) and refs[k].dtype == np.float64]
    data = np.squeeze(next(a for a in arrays if a.size == 6 * n and 6 in a.shape))
    data = data.T if data.shape == (6, n) else data.reshape(n, 6)
    tsim = next(a.ravel() for a in arrays if a.size == n)
    return tsim, data


def analyze_timing(r):
    """Where does the delay come from? Uses the A clock at every model step (t_A) and
    the B timestamp + B->A offset of every marker edge.
      lag      = t_A - model time; it jumps when the model stalls. After the jump the
                 model processes amplifier data that is `lag` old.
      base     = lag before the first stall (model in step with real time)
      sent_A   = edge_ts_B + offset = time the marker was pushed on B, on A's clock
    Per flash:
      transit  = t_A of the step that received the marker - sent_A  (LSL B -> Simulink)
      delay    = photodiode sample - marker sample in the EEG file (as in cells 2-4)
      backlog  = lag around the flash - base
      residual = photodiode sample time - (sent_A - base): what remains after placing
                 the marker by its timestamp instead of by the step that received it"""
    n, fs, t = r["shape"][0], r["fs"], r["t"]
    tsim, d = load_timing(r["timing_path"], n)
    assert np.allclose(tsim, t, atol=1e-6), "timing file and EEG file have different time axes"
    tA, off = d[:, 4], d[:, 3]
    lag = tA - tsim
    gaps = np.diff(tA)
    stall_idx = np.flatnonzero(gaps > STALL_MS / 1000)
    first = stall_idx[0] if stall_idx.size else n - 1
    lo = min(int(fs), first // 2)                       # skip the start-up second if possible
    base = np.median(lag[lo:first + 1])
    lag_ms = (lag - base) * 1000

    e = np.flatnonzero(d[:, 1] != 0)
    edges = pd.DataFrame({"step": e, "code": d[e, 1].astype(int), "sim_s": tsim[e],
                          "ts_B": d[e, 2], "sent_A": d[e, 2] + off[e]})
    t_start = edges.loc[edges.code == 9, "sim_s"].min()  # START marker

    def med(i0, i1):
        i0, i1 = max(0, i0), min(n, i1)
        return np.median(lag_ms[i0:i1]) if i1 > i0 else np.nan

    w = int(fs)
    stalls = pd.DataFrame({
        "sim_s": tsim[stall_idx],
        "s_before_START": t_start - tsim[stall_idx],
        "gap_ms": gaps[stall_idx] * 1000,
        # shift that is still there 2-3 s later (not caught up) vs. 1-2 s before
        "kept_ms": [med(i + 2 * w, i + 3 * w) - med(i - 2 * w, i - w) for i in stall_idx]})

    # pair every analysed flash with the timestamp-block edge closest to its marker
    p = r["pairs"].sort_values("flash").reset_index(drop=True)
    s_mk = np.searchsorted(t, p.onset_s_mk.to_numpy())
    s_pd = np.searchsorted(t, p.onset_s_pd.to_numpy())
    fl = edges[edges.code == FLASH_CODE].reset_index(drop=True)
    fsteps = fl.step.to_numpy()
    j = np.abs(fsteps[None, :] - s_mk[:, None]).argmin(axis=1)
    ok_pair = np.abs(fsteps[j] - s_mk) < 0.2 * fs
    f = fl.iloc[j].reset_index(drop=True)
    flash = pd.DataFrame({
        "flash": p.flash.to_numpy(),
        "transit_ts_ms": (tA[f.step] - f.sent_A) * 1000,
        "transit_hisys_ms": (tA[s_mk] - f.sent_A) * 1000,
        "delay_ms": p.delay_ms.to_numpy(),
        "backlog_ms": [med(s - w // 10, s + w // 10) for s in s_mk],
        "residual_ms": (t[s_pd] - (f.sent_A - base)) * 1000,
        "ts_B": f.ts_B.to_numpy()})[ok_pair].reset_index(drop=True)
    flash["delay_minus_backlog_ms"] = flash.delay_ms - flash.backlog_ms

    # which PsychoPy run on computer B sent these markers? (same LSL timestamps)
    csv_name, csv_n, lock_ms, run_info = None, 0, np.nan, {}
    for path, c in psychopy_csvs.items():
        k = np.isin(np.round(c.lsl_push_timestamp.to_numpy(), 6), np.round(flash.ts_B.to_numpy(), 6)).sum()
        if k:
            csv_name, csv_n = path.name, int(k)
            lock_ms = c["lock_wait_ms"].mean() if "lock_wait_ms" in c else np.nan
            info_path = path.with_name(path.name.replace("_latency_flashes.csv", "_run_info.txt"))
            if info_path.exists():                # "key: value" lines written by PsychoPy
                run_info = dict(l.split(": ", 1) for l in info_path.read_text(encoding="utf-8").splitlines()
                                if ": " in l)
            break
    resyncs = float(run_info["lsl_resyncs"]) if run_info.get("lsl_resyncs", "None") != "None" else np.nan

    main = stalls.loc[stalls.gap_ms.idxmax()] if len(stalls) else None
    r.update(ts_lag_ms=lag_ms, ts_tsim=tsim, ts_edges=edges, ts_stalls=stalls,
             ts_flash=flash, ts_start=t_start, ts_npulled=d[:, 5])
    s = {"backlog_ms": flash.backlog_ms.median(),
         "main_stall_ms": main.gap_ms if main is not None else 0.0,
         "main_stall_s_before_START": main.s_before_START if main is not None else np.nan,
         "stalls_over_100ms": len(stalls),
         "stalls_kept_over_50ms": int((stalls.kept_ms > 50).sum()),
         "transit_ts_mean_ms": flash.transit_ts_ms.mean(), "transit_ts_max_ms": flash.transit_ts_ms.max(),
         "transit_hisys_mean_ms": flash.transit_hisys_ms.mean(),
         "delay_ms": flash.delay_ms.mean(),
         "delay_minus_backlog_ms": flash.delay_minus_backlog_ms.mean(),
         "residual_mean_ms": flash.residual_ms.mean(), "residual_sd_ms": flash.residual_ms.std(),
         "lsl_samples_per_s": d[:, 5].sum() / (tsim[-1] - tsim[0]),
         "flashes_with_timestamp": len(flash),
         "psychopy_csv": csv_name, "csv_timestamps_matched": csv_n, "psychopy_lock_wait_ms": lock_ms,
         "psychopy_script": run_info.get("script", "run_photodiode_latency_v0.py (no script line)"),
         "lsl_resyncs": resyncs}
    r["ts_stats"] = s

    print(f"\n--- LSL timestamps ({r['timing_path'].name}) ---")
    print(f"PsychoPy run on computer B: {csv_name}  ({csv_n} of {len(flash)} flash timestamps identical)")
    print(f"PsychoPy script: {s['psychopy_script']}   lsl_resyncs: {resyncs:g}")
    print(f"LSL samples received: {s['lsl_samples_per_s']:.1f} per s of model time")
    print(f"Stalls (model step > {STALL_MS} ms): {len(stalls)}")
    if len(stalls):
        print(stalls.sort_values("gap_ms", ascending=False).head(5).round(2).to_string(index=False))
    print(f"Backlog during the flashes: {s['backlog_ms']:.1f} ms  "
          f"(largest stall {s['main_stall_ms']:.0f} ms, {s['main_stall_s_before_START']:.2f} s before the START marker)")
    print(f"Marker transit B -> Simulink: timestamp block {s['transit_ts_mean_ms']:.2f} ms "
          f"(max {s['transit_ts_max_ms']:.1f}), g.HIsys block {s['transit_hisys_mean_ms']:.2f} ms")
    print(f"Photodiode - marker in the EEG: {s['delay_ms']:.1f} ms = backlog {s['backlog_ms']:.1f} ms "
          f"+ {s['delay_minus_backlog_ms']:.1f} ms")
    print(f"Photodiode vs. timestamp-placed marker: {s['residual_mean_ms']:.1f} ms, SD {s['residual_sd_ms']:.2f} ms")


# Cell 3: analyse every recording, results side by side

Recordings with an `*_LSLtiming.mat` file also print the LSL timestamp results.

In [ ]:
for r in recs:
    print(f"\n{'=' * 25} {r['label']} {'=' * 25}\n{r['path'].name}\n")
    try:
        detect(r)
        print()
        pair_flashes(r)
        r["stats"] = summarize(r)
        r["ok"] = True
    except Exception as e:                        # one bad recording must not stop the others
        r["ok"] = False
        print(f"!! analysis failed: {type(e).__name__}: {e}")
        continue
    if r["timing_path"] is not None:
        try:
            analyze_timing(r)
        except Exception as e:
            print(f"!! LSL timestamp analysis failed: {type(e).__name__}: {e}")

ok = [r for r in recs if r["ok"]]
failed = [r["label"] for r in recs if not r["ok"]]
assert ok, "No recording could be analysed."
summary = pd.DataFrame([{**r["cfg"], **r["stats"]} for r in ok], index=[r["label"] for r in ok])

print(f"\n{'=' * 25} RESULTS SIDE BY SIDE {'=' * 25}")
if failed:
    print(f"Left out (analysis failed): {failed}")
print(summary.round(2).T.to_string())

# Cell 4: width, delay and jitter plots side by side

One column per recording. The width row and the jitter row use the same scale in every
column; the delay row keeps its own scale (cell 6 overlays the delays on one scale).

In [ ]:
def same_limits(axs, which):
    get, set_ = ("get_ylim", "set_ylim") if which == "y" else ("get_xlim", "set_xlim")
    lims = np.array([getattr(a, get)() for a in axs])
    for a in axs:
        getattr(a, set_)(lims[:, 0].min(), lims[:, 1].max())


n = len(ok)
fig, axes = plt.subplots(3, n, figsize=(5.5 * n, 12), squeeze=False)
for j, r in enumerate(ok):
    mk, pairs, s = r["mk"], r["pairs"], r["stats"]
    ax = axes[0, j]
    ax.plot(mk.flash, mk.intended_ms, "k--", lw=1, label="intended")
    ax.plot(mk.flash, mk.width_ms, ".", label="marker")
    ax.plot(pairs.flash, pairs.width_ms_pd, ".", label="photodiode")
    ax.set(xlabel="flash number", ylabel="pulse width (ms)",
           title=f"{r['label']}\nRecorded width per flash")
    ax = axes[1, j]
    ax.plot(pairs.flash, pairs.delay_ms, ".")
    ax.axhline(s["latency_mean_ms"], color="k", lw=0.8, ls=":")
    ax.set(xlabel="flash number", ylabel="delay (ms)",
           title=f"Photodiode onset - marker onset\nmean {s['latency_mean_ms']:.1f} ms, "
                 f"drift {s['drift_ms_per_min']:+.1f} ms/min")
    ax = axes[2, j]
    ax.hist(r["detr"], bins=30)
    ax.set(xlabel="delay after removing drift (ms)", ylabel="flashes",
           title=f"Jitter: SD {s['jitter_sd_detrended_ms']:.2f} ms")
axes[0, 0].legend()
same_limits(axes[0], "y")
same_limits(axes[2], "x")
plt.tight_layout(); plt.show()

# Cell 5: raw marker and photodiode traces side by side

Rows are the same time windows as in the single-recording notebook; columns are the
recordings.

In [ ]:
def trace_windows(r):
    t, ev, mk, pairs = r["t"], r["ev"], r["mk"], r["pairs"]
    mid = pairs.onset_s_mk.iloc[len(pairs) // 2]
    cal = ev.loc[ev.code == CAL_DARK, "onset_s"].min()
    start = (cal if np.isfinite(cal) else mk.onset_s.iloc[0]) - 1
    return [("Whole recording", t[0], t[-1]),
            ("Calibration and first flashes", start, mk.onset_s.iloc[0] + 6),
            ("Middle of the flash block", mid - 1, mid + 9),
            ("Last flashes", mk.onset_s.iloc[-1] - 9, min(t[-1], mk.onset_s.iloc[-1] + 4))]


n_win = 4
fig, axes = plt.subplots(n_win, len(ok), figsize=(6 * len(ok), 3 * n_win), squeeze=False)
for j, r in enumerate(ok):
    t, marker, ev, pairs = r["t"], r["marker"], r["ev"], r["pairs"]
    photo01 = r["square_on"].astype(int)
    for i, (title, t0, t1) in enumerate(trace_windows(r)):
        ax = axes[i, j]
        w = (t >= t0) & (t <= t1)
        ax.fill_between(t[w], 0, photo01[w], step="post", color="gold", alpha=0.6, label="photodiode (square on)")
        ax.fill_between(t[w], 1.2, 1.2 + (marker[w] > 0) * 0.8, step="post", color="tab:blue", alpha=0.6, label="marker")
        if t1 - t0 < 60:
            for _, e in ev[(ev.onset_s >= t0) & (ev.onset_s <= t1) & (ev.code != FLASH_CODE)].iterrows():
                ax.text(e.onset_s, 2.05, str(int(e.code)), color="tab:blue", fontsize=8)
            for _, p in pairs[(pairs.onset_s_mk >= t0) & (pairs.onset_s_mk <= t1)].iterrows():
                ax.plot([p.onset_s_mk, p.onset_s_pd], [1.2, 1.0], "k-", lw=0.8)
                ax.text(p.onset_s_mk, 2.05, str(int(p.flash)), fontsize=7)
        ax.set(title=f"{r['label']}: {title}" if i == 0 else title,
               xlim=(t0, t1), ylim=(-0.05, 2.3), yticks=[], xlabel="recording time (s)")
axes[0, 0].legend(loc="upper left", ncol=2)
plt.tight_layout(); plt.show()

# Cell 6: comparison between the recordings

- **Latency** and **jitter** per recording (`COMPARE_BY = None`, the default), or against a
  file-name number such as `COMPARE_BY = "slx_frame_length"` (set in cell 1).
- With `COMPARE_BY = "slx_frame_length"`: `frame_ms` = one Simulink frame (frame length / fs).
  If the markers are read once per frame, the delay is spread evenly over one frame, which
  adds about `frame_ms / sqrt(12)` of jitter (dashed line in the jitter plot).
- **Time base** = recorded width increase per flash / intended increase. 1.000 means the
  Simulink model kept real time; below 1 means it ran slow and the recording is compressed.

In [ ]:
cmp = summary.copy()
xkey = COMPARE_BY
numeric = xkey in cmp and cmp[xkey].notna().all()
x = cmp[xkey].to_numpy(float) if numeric else np.arange(len(cmp), dtype=float)
xlabel = xkey if numeric else "recording"
if numeric:
    cmp.insert(cmp.columns.get_loc(xkey) + 1, "frame_ms", cmp[xkey] / cmp.fs_hz * 1000)

cols = [xkey, "frame_ms", "fs_hz", "paired", "missing", "latency_mean_ms", "latency_median_ms",
        "jitter_sd_ms", "jitter_sd_detrended_ms", "drift_ms_per_min", "delay_min_ms", "delay_max_ms",
        "timebase_marker", "timebase_photodiode"]
table = cmp[[c for c in cols if c in cmp]].copy()
ref = table.iloc[0]
table["latency_vs_first_ms"] = table.latency_mean_ms - ref.latency_mean_ms
table["jitter_vs_first_ms"] = table.jitter_sd_detrended_ms - ref.jitter_sd_detrended_ms
print(f"Compared to the first recording ({table.index[0]}):\n")
print(table.round(2).to_string())

if numeric and cmp[xkey].nunique() >= 2:
    sl_lat = np.polyfit(x, cmp.latency_mean_ms, 1)[0]
    sl_jit = np.polyfit(x, cmp.jitter_sd_detrended_ms, 1)[0]
    step_ms = 1000 / cmp.fs_hz.median()
    print(f"\nStraight-line fit across recordings, per +1 {xkey}:")
    print(f"  latency {sl_lat:+.3f} ms   jitter SD {sl_jit:+.3f} ms   (one sample = {step_ms:.2f} ms)")
    best = cmp.jitter_sd_detrended_ms.idxmin()
    print(f"  lowest jitter: {best}  (SD {cmp.loc[best, 'jitter_sd_detrended_ms']:.2f} ms, "
          f"latency {cmp.loc[best, 'latency_mean_ms']:.1f} ms)")


def x_axis(ax):
    if numeric:
        ax.set_xticks(uniq)
    else:
        ax.set_xticks(x, cmp.index, rotation=30, ha="right")
    ax.set_xlabel(xlabel)


def mean_line(ax, y):                             # thin grey line through the mean of each setting
    y = np.asarray(y, float)
    ax.plot(uniq, [y[x == u].mean() for u in uniq], "-", lw=0.8, color="grey", zorder=0)


# recordings with the same setting are spread slightly apart so they do not overlap
uniq = np.unique(x)
spacing = np.min(np.diff(uniq)) if len(uniq) > 1 else 1.0
xp = x.copy()
for u in uniq:
    idx = np.flatnonzero(x == u)
    xp[idx] = u + (np.arange(len(idx)) - (len(idx) - 1) / 2) * 0.25 * spacing
w = 0.2 * spacing
cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
colors = [cycle[k % len(cycle)] for k in range(len(ok))]
shape_legend = lambda *items: [plt.Line2D([], [], color="grey", ls="none", **kw, label=l) for l, kw in items]

fig, ax = plt.subplots(2, 3, figsize=(18, 10))

a = ax[0, 0]
a.vlines(xp, cmp.delay_min_ms, cmp.delay_max_ms, colors=colors, lw=8, alpha=0.2)
for k in range(len(ok)):
    a.errorbar(xp[k], cmp.latency_mean_ms.iloc[k], yerr=cmp.jitter_sd_ms.iloc[k],
               fmt="o", capsize=4, color=colors[k])
a.plot(xp, cmp.latency_median_ms, "k_", ms=12)
mean_line(a, cmp.latency_mean_ms)
a.legend(handles=shape_legend(("mean ± SD", {"marker": "o"}), ("median", {"marker": "_", "ms": 12}),
                              ("min to max", {"marker": "|", "ms": 14, "alpha": 0.4})))
a.set(ylabel="delay (ms)", title="Latency (photodiode onset - marker onset)"); x_axis(a)

a = ax[0, 1]
for k in range(len(ok)):
    a.plot(xp[k], cmp.jitter_sd_ms.iloc[k], "o", mfc="none", color=colors[k])
    a.plot(xp[k], cmp.jitter_sd_detrended_ms.iloc[k], "s", color=colors[k])
mean_line(a, cmp.jitter_sd_detrended_ms)
handles = shape_legend(("SD raw", {"marker": "o", "mfc": "none"}), ("SD after removing drift", {"marker": "s"}))
if numeric:
    handles += a.plot(uniq, uniq / cmp.fs_hz.median() * 1000 / np.sqrt(12), "k--", lw=1,
                      label="frame_ms / sqrt(12)")
a.legend(handles=handles)
a.set(ylabel="SD (ms)", title="Jitter"); x_axis(a)

a = ax[0, 2]
for k in range(len(ok)):
    a.plot(xp[k], cmp.timebase_marker.iloc[k], "o", mfc="none", color=colors[k])
    a.plot(xp[k], cmp.timebase_photodiode.iloc[k], "s", color=colors[k])
a.axhline(1, color="k", lw=0.8, ls=":")
a.legend(handles=shape_legend(("marker", {"marker": "o", "mfc": "none"}), ("photodiode", {"marker": "s"})))
a.set(ylabel="recorded / intended width step", title="Time base (1 = model keeps real time)"); x_axis(a)

a = ax[1, 0]
for k, r in enumerate(ok):
    a.plot(r["pairs"].flash, r["pairs"].delay_ms, ".-", lw=0.6, ms=4, color=colors[k], label=r["label"])
a.set(xlabel="flash number", ylabel="delay (ms)", title="Delay per flash, all recordings (colours used in every panel)")
a.legend(fontsize=8)

a = ax[1, 1]
bp = a.boxplot([r["detr"] for r in ok], positions=xp, widths=w, patch_artist=True, manage_ticks=False)
for patch, c in zip(bp["boxes"], colors):
    patch.set(facecolor=c, alpha=0.5)
a.set(ylabel="delay after removing drift (ms)", title="Jitter distribution"); x_axis(a)

a = ax[1, 2]
a.bar(xp, cmp.drift_ms_per_min, width=w, color=colors)
a.axhline(0, color="k", lw=0.8)
a.set(ylabel="ms per minute", title="Drift of the delay over the run"); x_axis(a)

plt.tight_layout(); plt.show()

# Cell 7: LSL timestamps, side by side (timestamp model only)

One column per recording made with the timestamp model.

- **Top:** how far the model is behind real time (`t_A` − model time, relative to the start).
  A step up that stays up is a stall the model never caught up; it shifts all amplifier data,
  and so the photodiode, later relative to the markers. Red dots = stalls > `STALL_MS`;
  dashed line = START marker (code 9) from PsychoPy.
- **Middle:** per flash, photodiode − marker in the EEG (as in cells 2-4), the backlog at that
  flash, and what is left after removing the backlog.
- **Bottom:** time from the push on computer B to the Simulink step that received the marker
  (LSL transit), for the timestamp block and the g.HIsys LSL Receive block.

In [ ]:
ts_recs = [r for r in ok if "ts_stats" in r]
assert ts_recs, "No recording with an analysed *_LSLtiming.mat file."
n = len(ts_recs)
fig, axes = plt.subplots(3, n, figsize=(5.5 * n, 12), squeeze=False)
for j, r in enumerate(ts_recs):
    s, fl, st = r["ts_stats"], r["ts_flash"], r["ts_stalls"]
    ax = axes[0, j]
    k = slice(None, None, 10)                                   # every 10th step is enough
    ax.plot(r["ts_tsim"][k], r["ts_lag_ms"][k], lw=0.8)
    if len(st):
        ax.plot(st.sim_s, np.interp(st.sim_s, r["ts_tsim"], r["ts_lag_ms"]), "r.", ms=6)
    ax.axvline(r["ts_start"], color="k", ls="--", lw=0.8)
    ax.set(xlabel="model time (s)", ylabel="model behind real time (ms)",
           title=f"{r['label']}\nbacklog {s['backlog_ms']:.0f} ms; largest stall "
                 f"{s['main_stall_s_before_START']:.1f} s before START")
    ax = axes[1, j]
    ax.plot(fl.flash, fl.delay_ms, ".", label="photodiode - marker (EEG)")
    ax.plot(fl.flash, fl.backlog_ms, ".", label="backlog")
    ax.plot(fl.flash, fl.delay_minus_backlog_ms, ".", label="difference")
    ax.set(xlabel="flash number", ylabel="ms",
           title=f"delay {s['delay_ms']:.0f} ms = backlog {s['backlog_ms']:.0f} + "
                 f"{s['delay_minus_backlog_ms']:.0f} ms")
    ax = axes[2, j]
    bins = np.linspace(0, max(5, np.percentile(np.r_[fl.transit_ts_ms, fl.transit_hisys_ms], 99)), 40)
    ax.hist(fl.transit_ts_ms, bins=bins, alpha=0.6, label="timestamp block")
    ax.hist(fl.transit_hisys_ms, bins=bins, alpha=0.6, label="g.HIsys block")
    ax.set(xlabel="push on B -> Simulink step (ms)", ylabel="flashes",
           title=f"LSL transit: mean {s['transit_ts_mean_ms']:.2f} ms, max {s['transit_ts_max_ms']:.1f} ms")
axes[1, 0].legend(fontsize=8)
axes[2, 0].legend(fontsize=8)
same_limits(axes[0], "y")
same_limits(axes[1], "y")
plt.tight_layout(); plt.show()

# Cell 8: comparison of the timestamp runs, and what it means

- `backlog_ms`: how far the amplifier data is behind when the flashes are shown.
- `delay_minus_backlog_ms`: photodiode − marker once the backlog is removed (display +
  amplifier latency).
- `residual_mean_ms` / `residual_sd_ms`: photodiode onset relative to the marker placed by
  its LSL timestamp. This needs no photodiode, so if it is the same in every run, its mean is
  the calibration constant for the real experiment and its SD the remaining jitter.
- `stalls_kept_over_50ms`: stalls that the model never caught up.
- `lsl_resyncs` (from the PsychoPy `*_run_info.txt`): pauses > 0.1 s in the marker stream.
- The conclusions are printed separately for the v0 sender and for the fixed senders
  (changes 9 and 10 in the file name).

In [ ]:
tcols = ["main_stall_ms", "main_stall_s_before_START", "stalls_over_100ms", "stalls_kept_over_50ms",
         "backlog_ms", "delay_ms", "delay_minus_backlog_ms", "residual_mean_ms", "residual_sd_ms",
         "transit_ts_mean_ms", "transit_ts_max_ms", "transit_hisys_mean_ms", "lsl_samples_per_s",
         "flashes_with_timestamp", "lsl_resyncs", "psychopy_csv", "csv_timestamps_matched",
         "psychopy_lock_wait_ms", "psychopy_script"]
tsum = pd.DataFrame([r["ts_stats"] for r in ts_recs], index=[r["label"] for r in ts_recs])[tcols]
print(tsum.drop(columns=["psychopy_csv", "psychopy_script"]).round(2).T.to_string())
print("\nPsychoPy runs:")
for lab, row in tsum.iterrows():
    print(f"  {lab}: {row.psychopy_csv}  ({row.psychopy_script})")

colors = [plt.rcParams["axes.prop_cycle"].by_key()["color"][k % 10] for k in range(len(ts_recs))]
xi = np.arange(len(ts_recs))
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
a = ax[0]
a.bar(xi, tsum.backlog_ms, color="tab:red", alpha=0.7, label="backlog (model stall)")
a.bar(xi, tsum.delay_minus_backlog_ms, bottom=tsum.backlog_ms, color="tab:blue", alpha=0.7,
      label="rest (display + amplifier)")
a.set(ylabel="photodiode - marker (ms)", title="Where the delay comes from"); a.legend()
a = ax[1]
a.errorbar(xi, tsum.residual_mean_ms, yerr=tsum.residual_sd_ms, fmt="o", capsize=5, color="k")
a.axhline(tsum.residual_mean_ms.mean(), color="grey", ls=":", lw=1)
a.set(ylabel="ms", title="Photodiode vs. timestamp-placed marker (mean ± SD)")
a = ax[2]
a.bar(xi - 0.2, tsum.main_stall_ms, width=0.4, label="largest stall (ms)")
a.bar(xi + 0.2, tsum.main_stall_s_before_START * 1000, width=0.4, label="its time before START (ms)")
a.set(title="The stall that creates the backlog"); a.legend()
for a in ax:
    a.set_xticks(xi, tsum.index, rotation=30, ha="right")
plt.tight_layout(); plt.show()

# --- conclusions computed from the table ------------------------------------------------
is_fixed = np.array([any(r["cfg"].get(k) for k in SENDER_FLAGS) for r in ts_recs])
old, new = tsum[~is_fixed], tsum[is_fixed]
print("\n=== What the timestamp runs show ===")
print("1. LSL transit B -> Simulink (push on B to the Simulink step that received the marker), mean / max:")
for lab, row in tsum.iterrows():
    print(f"     {lab}: timestamp block {row.transit_ts_mean_ms:.1f} / {row.transit_ts_max_ms:.0f} ms, "
          f"g.HIsys block {row.transit_hisys_mean_ms:.1f} ms")
if len(old):
    res = old.residual_mean_ms
    print(f"\n--- v0 sender ({len(old)} runs: outlet created before the window) ---")
    print(f"2. Photodiode - marker ({old.delay_ms.min():.0f}-{old.delay_ms.max():.0f} ms) = backlog "
          f"({old.backlog_ms.min():.0f}-{old.backlog_ms.max():.0f} ms) + "
          f"{old.delay_minus_backlog_ms.min():.0f}-{old.delay_minus_backlog_ms.max():.0f} ms.")
    print(f"3. The backlog comes from one stall of {old.main_stall_ms.min():.0f}-{old.main_stall_ms.max():.0f} ms, "
          f"{old.main_stall_s_before_START.min():.2f}-{old.main_stall_s_before_START.max():.2f} s before the START "
          "marker in every run, including the run started 1 min after the model; "
          f"it is the only stall the model never catches up ({old.stalls_kept_over_50ms.max():.0f} kept per run, "
          f"lsl_resyncs {old.lsl_resyncs.min():g}-{old.lsl_resyncs.max():g}).")
    print("   No LSL samples arrive during it either, so the model was waiting for the marker stream "
          "(it takes one LSL sample per step). It starts at the first flips of the new fullscreen window "
          "(start of getActualFrameRate in the PsychoPy log). After a pause longer than _MAX_CATCHUP_S "
          "(0.1 s), lsl_trigger.py restarts its sample count instead of sending the missed samples, so "
          "the model stays behind.")
    print(f"4. Placing each marker by its LSL timestamp leaves {res.min():.1f}-{res.max():.1f} ms "
          f"(mean {res.mean():.1f} ms, between-run SD {res.std():.1f} ms; within-run SD "
          f"{old.residual_sd_ms.min():.1f}-{old.residual_sd_ms.max():.1f} ms).")
    print("5. Start delay, workspace logging off and multithreading off change the delay only through "
          "the size of that one stall.")
if len(new):
    print(f"\n--- fixed sender v1 / v2 ({len(new)} runs: outlet created after the window setup) ---")
    for lab, row in new.iterrows():
        print(f"6. {lab}: photodiode - marker {row.delay_ms:.1f} ms, backlog {row.backlog_ms:.1f} ms, "
              f"stalls kept {row.stalls_kept_over_50ms:.0f}, lsl_resyncs {row.lsl_resyncs:g}, "
              f"timestamp residual {row.residual_mean_ms:.1f} ± {row.residual_sd_ms:.1f} ms")
    if len(old):
        print(f"   Compared with v0: photodiode - marker {old.delay_ms.mean():.0f} -> {new.delay_ms.mean():.0f} ms "
              f"(v0 without the backlog: {old.delay_minus_backlog_ms.mean():.0f} ms). The stall is gone.")
        print(f"7. LSL transit rose from {old.transit_ts_mean_ms.mean():.1f} ms (v0) to "
              f"{new.transit_ts_mean_ms.mean():.1f} ms (timestamp block) / {new.transit_hisys_mean_ms.mean():.1f} ms "
              "(g.HIsys block), spread evenly over 0 to about two amplifier frames. Likely reason: without a "
              "backlog the model waits for the amplifier and processes each delivered frame "
              "(slx_frame_length samples) in a burst, so a marker waits for the next burst. In v0 the "
              "0.8 s backlog meant amplifier data was always ready, so the model ran at the pace of the "
              "LSL samples and picked each marker up at once. The recorded marker is therefore later "
              "than before, which is why photodiode - marker is now below the v0 value without backlog.")
        print(f"8. The timestamp-placed residual is unaffected: v0 {old.residual_mean_ms.min():.1f}-"
              f"{old.residual_mean_ms.max():.1f} ms, fixed sender {new.residual_mean_ms.min():.1f}-"
              f"{new.residual_mean_ms.max():.1f} ms (all runs: mean {tsum.residual_mean_ms.mean():.1f} ms, "
              f"between-run SD {tsum.residual_mean_ms.std():.1f} ms).")